In [1]:
import pandas as pd
from rapidfuzz import fuzz
import os


In [2]:
df=pd.read_csv(r"C:\Users\Asus\Downloads\Jupiter Py Pro\IMDB Project\ML- Layer\imdb_enriched.csv")
df.tail()

,name,year,match_method,omdb_title,omdb_year,Plot,Genre_full,Actors_full,Writer_full,Director_full,imdbRating_omdb
4684,Corpse Bride,2005,exact,Corpse Bride,2005,Set back in the late 1800s in a Victorian vill...,"Animation, Comedy, Drama","Johnny Depp, Helena Bonham Carter, Emily Watson","Tim Burton, Carlos Grangel, John August","Tim Burton, Mike Johnson",7.4
4685,Transporter 2,2005,exact,Transporter 2,2005,Frank Martin is the best in the business. The ...,"Action, Thriller","Jason Statham, Amber Valletta, Kate Nauta","Luc Besson, Robert Mark Kamen",Louis Leterrier,6.3
4686,Two for the Money,2005,exact,Two for the Money,2005,Brandon Lang loves football: an injury keeps h...,"Crime, Drama, Sport","Matthew McConaughey, Al Pacino, Rene Russo",Dan Gilroy,D.J. Caruso,6.2
4687,The New World,2005,exact,The New World,2005,Captain Smith is spared his mutinous hanging s...,"Biography, Drama, History","Colin Farrell, Q'orianka Kilcher, Christopher ...",Terrence Malick,Terrence Malick,6.7
4688,Brick,2005,exact,Brick,2005,The lonely teenager Brendan finds his former g...,"Crime, Drama, Mystery","Joseph Gordon-Levitt, Lukas Haas, Emilie de Ravin",Rian Johnson,Rian Johnson,7.1


In [3]:
TITLE_SIM_FLOOR = 55       
YEAR_DIFF_CEILING = 1
BAD_GENRES = {"talk-show", "short", "episode", "documentary short", "game-show", "reality-tv"}
def flag_reason(row):
    """Return a reason string if the row is bad, else None."""
    if pd.isna(row["Plot"]) or str(row["Plot"]).strip() in ("", "N/A"):
        return "missing plot"

    title_sim = fuzz.token_sort_ratio(str(row["name"]), str(row["omdb_title"]))
    if title_sim < TITLE_SIM_FLOOR:
        return f"title mismatch ({title_sim:.0f})"

    try:
        year_diff = abs(int(row["year"]) - int(str(row["omdb_year"])[:4]))
        if year_diff > YEAR_DIFF_CEILING:
            return f"year mismatch ({row['year']} vs {row['omdb_year']})"
    except (ValueError, TypeError):
        return "bad year"

    genres = str(row.get("Genre_full", "")).lower()
    if any(g in genres for g in BAD_GENRES):
        return f"non-movie genre ({row['Genre_full']})"

    return None


In [4]:
df.head()

,name,year,match_method,omdb_title,omdb_year,Plot,Genre_full,Actors_full,Writer_full,Director_full,imdbRating_omdb
0,The Shining,1980,exact,The Shining,1980,"Haunted by a persistent writer's block, the as...","Drama, Horror","Jack Nicholson, Shelley Duvall, Danny Lloyd","Stephen King, Stanley Kubrick, Diane Johnson",Stanley Kubrick,8.4
1,The Blue Lagoon,1980,exact,The Blue Lagoon,1980,"On a journey to San Francisco, Richard, his fa...","Adventure, Drama, Romance","Brooke Shields, Christopher Atkins, Leo McKern","Henry De Vere Stacpoole, Douglas Day Stewart",Randal Kleiser,5.8
2,Star Wars: Episode V - The Empire Strikes Back,1980,exact,Star Wars: Episode V - The Empire Strikes Back,1980,"Luke Skywalker, Han Solo, Princess Leia and Ch...","Action, Adventure, Fantasy","Mark Hamill, Harrison Ford, Carrie Fisher","Leigh Brackett, Lawrence Kasdan, George Lucas",Irvin Kershner,8.7
3,Airplane!,1980,exact,Airplane!,1980,Drowning his sorrows after that botched missio...,Comedy,"Robert Hays, Julie Hagerty, Leslie Nielsen","Jim Abrahams, David Zucker, Jerry Zucker","Jim Abrahams, David Zucker, Jerry Zucker",7.7
4,Caddyshack,1980,exact,Caddyshack,1980,There's something fishy going on at the elitis...,"Comedy, Sport","Chevy Chase, Rodney Dangerfield, Bill Murray","Brian Doyle-Murray, Harold Ramis, Douglas Kenney",Harold Ramis,7.2


In [5]:
# Dropping Rows with insufficient data Using function we created before
reasons = df.apply(flag_reason, axis=1)
dropped = df[reasons.notna()].copy()
dropped["drop_reason"] = reasons[reasons.notna()]
clean = df[reasons.isna()].copy()
dropped.head()
print(f"Dropping {len(dropped)} of {len(df)} rows:")
print(dropped["drop_reason"].value_counts())

Dropping 46 of 4689 rows:
drop_reason
missing plot                            24
title mismatch (55)                      3
title mismatch (33)                      2
title mismatch (54)                      2
title mismatch (45)                      1
year mismatch (1986 vs 2011)             1
year mismatch (1988 vs 1990)             1
year mismatch (1989 vs 1984)             1
year mismatch (1992 vs 2026)             1
title mismatch (36)                      1
year mismatch (1993 vs 2019)             1
year mismatch (1993 vs 1995)             1
non-movie genre (Documentary, Short)     1
title mismatch (50)                      1
year mismatch (1999 vs 2002)             1
year mismatch (2000 vs 2019)             1
year mismatch (2001 vs 1999)             1
year mismatch (2001 vs 2003)             1
year mismatch (2002 vs 2004)             1
Name: count, dtype: int64


In [6]:
# Feature Engineering
clean["lead_actor"] = clean["Actors_full"].str.split(",").str[0].str.strip()
clean["Director"] = clean["Director_full"].str.split(",").str[0].str.strip()
clean["year"] = clean["omdb_year"].fillna(clean["year"])
clean['Plot']=clean["Plot"].str.strip()
clean.head()

,name,year,match_method,omdb_title,omdb_year,Plot,Genre_full,Actors_full,Writer_full,Director_full,imdbRating_omdb,lead_actor,Director
0,The Shining,1980,exact,The Shining,1980,"Haunted by a persistent writer's block, the as...","Drama, Horror","Jack Nicholson, Shelley Duvall, Danny Lloyd","Stephen King, Stanley Kubrick, Diane Johnson",Stanley Kubrick,8.4,Jack Nicholson,Stanley Kubrick
1,The Blue Lagoon,1980,exact,The Blue Lagoon,1980,"On a journey to San Francisco, Richard, his fa...","Adventure, Drama, Romance","Brooke Shields, Christopher Atkins, Leo McKern","Henry De Vere Stacpoole, Douglas Day Stewart",Randal Kleiser,5.8,Brooke Shields,Randal Kleiser
2,Star Wars: Episode V - The Empire Strikes Back,1980,exact,Star Wars: Episode V - The Empire Strikes Back,1980,"Luke Skywalker, Han Solo, Princess Leia and Ch...","Action, Adventure, Fantasy","Mark Hamill, Harrison Ford, Carrie Fisher","Leigh Brackett, Lawrence Kasdan, George Lucas",Irvin Kershner,8.7,Mark Hamill,Irvin Kershner
3,Airplane!,1980,exact,Airplane!,1980,Drowning his sorrows after that botched missio...,Comedy,"Robert Hays, Julie Hagerty, Leslie Nielsen","Jim Abrahams, David Zucker, Jerry Zucker","Jim Abrahams, David Zucker, Jerry Zucker",7.7,Robert Hays,Jim Abrahams
4,Caddyshack,1980,exact,Caddyshack,1980,There's something fishy going on at the elitis...,"Comedy, Sport","Chevy Chase, Rodney Dangerfield, Bill Murray","Brian Doyle-Murray, Harold Ramis, Douglas Kenney",Harold Ramis,7.2,Chevy Chase,Harold Ramis


In [7]:
# Renaming And getting cleaned data into new file
out = clean[["name", "year", "lead_actor", "Director", "imdbRating_omdb", "Genre_full", "Plot"]]
out = out.rename(columns={"imdbRating_omdb": "IMDB-Rating", "Genre_full": "Genre","name":"Name"})
out = out.rename(columns={
    "IMDB-Rating": "imdb_rating",
    "Genre": "genre",
    "Name": "name",          # already lowercase from source, no change needed
    "Director": "director",
    "Plot": "plot",
})
if os.path.exists("live_additions.csv"):
    live = pd.read_csv("live_additions.csv")
    out = pd.concat([out, live], ignore_index=True)
    out = out.drop_duplicates(subset=["name", "year"], keep="first")
    print(f"Merged in {len(live)} live-added movies")


Merged in 10 live-added movies


In [8]:
out.head()

,name,year,lead_actor,director,imdb_rating,genre,plot
0,The Shining,1980,Jack Nicholson,Stanley Kubrick,8.4,"Drama, Horror","Haunted by a persistent writer's block, the as..."
1,The Blue Lagoon,1980,Brooke Shields,Randal Kleiser,5.8,"Adventure, Drama, Romance","On a journey to San Francisco, Richard, his fa..."
2,Star Wars: Episode V - The Empire Strikes Back,1980,Mark Hamill,Irvin Kershner,8.7,"Action, Adventure, Fantasy","Luke Skywalker, Han Solo, Princess Leia and Ch..."
3,Airplane!,1980,Robert Hays,Jim Abrahams,7.7,Comedy,Drowning his sorrows after that botched missio...
4,Caddyshack,1980,Chevy Chase,Harold Ramis,7.2,"Comedy, Sport",There's something fishy going on at the elitis...


In [9]:
out.to_csv("Clean_Data.csv", index=False)
dropped.to_csv("dropped_rows.csv", index=False)
print(f"\n{len(out)} clean rows -> Clean_Data.csv")


4650 clean rows -> Clean_Data.csv
